In [ ]:
!pip install sentence-transformers pypdf transformers huggingface_hub torch tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import warnings
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import pipeline
from huggingface_hub import login
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from os import pipe
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
warnings.filterwarnings("ignore", category=UserWarning)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [ ]:
LLM_MODEL = "google/flan-t5-base"  # language model for generating answers
EMBEDDING_MODEL = "all-MiniLM-L12-v2"  # model for creating embeddings
HF_API_KEY = os.getenv("HF_API_KEY", "YOUR-HF-API-KEY-HERE")  # Optional Hugging Face API key

print(f"📋 LLM Model: {LLM_MODEL}")
print(f"📋 Embedding Model: {EMBEDDING_MODEL}")

📋 LLM Model: google/flan-t5-base
📋 Embedding Model: all-MiniLM-L12-v2


Initialize the Vector Database

In [ ]:
class VectorDB:
    def __init__(self, model_name: str):
        
        self.embedModel = SentenceTransformer(EMBEDDING_MODEL)
        self.embed_size = self.embedModel.get_sentence_embedding_dimension()
        self._embeddings = np.empty((0, self.embed_size))
        self._strings = []

        print(f"✅ VectorDB initialized with embedding dimension: {self.embed_size}")

In [ ]:
def addToDatabase(self, input: list[str]):
    
    new_embeddings = self.embedModel.encode(input)

    if self._embeddings.shape[0] == 0:
        self._embeddings = new_embeddings
    else:
        self._embeddings = np.vstack((self._embeddings, new_embeddings))

    list.extend(self._strings, input)

    print(f"✅ Added {len(input)} chunks. Total chunks: {len(self._strings)}")

VectorDB.addToDatabase = addToDatabase

In [ ]:
def clearDatabase(self):
    
    self._embeddings = np.empty((0, self.embed_size))
    self._strings = []

    print("🗑️ Database cleared!")

VectorDB.clearDatabase = clearDatabase

In [ ]:
def euclideanSim(self, x, y):
   
    distance = np.linalg.norm(x - y)
    similarity = 1 / (1 + distance)

    return similarity

VectorDB.euclideanSim = euclideanSim

In [ ]:
def search(self, query: str, n_return=3):
   
    query_embedding = self.embedModel.encode([query])[0]
    similarities = [self.euclideanSim(query_embedding, emb) for emb in self._embeddings]
    top_indices = np.argsort(similarities)[::-1][:n_return]

    top_chunks = [self._strings[i] for i in top_indices]
    top_scores = [similarities[i] for i in top_indices]

    return top_chunks, top_scores

VectorDB.search = search

In [ ]:
print("Testing VectorDB...\n")

test_vdb = VectorDB(EMBEDDING_MODEL)
test_data = [
    "Python is a programming language.",
    "Machine learning involves training models on data.",
    "Dogs are loyal pets.",
    "Neural networks are inspired by the human brain."
]

test_vdb.addToDatabase(test_data)

query = "What is ML?"
results, scores = test_vdb.search(query, n_return=2)

print(f"\n🔍 Query: '{query}'\n")
for i, (chunk, score) in enumerate(zip(results, scores), 1):
    print(f"Result {i} (similarity: {score:.4f}):")
    print(f"  {chunk}\n")

Testing VectorDB...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ VectorDB initialized with embedding dimension: 384
✅ Added 4 chunks. Total chunks: 4

🔍 Query: 'What is ML?'

Result 1 (similarity: 0.4539):
  Machine learning involves training models on data.

Result 2 (similarity: 0.4379):
  Neural networks are inspired by the human brain.



In [ ]:
def clean_text(text: str) -> str:
    return " ".join(text.split())

test_text = "This   has    extra\n\nspaces   and\nnewlines."
print(f"Original: {repr(test_text)}")
print(f"Cleaned:  {repr(clean_text(test_text))}")

Original: 'This   has    extra\n\nspaces   and\nnewlines.'
Cleaned:  'This has extra spaces and newlines.'


In [ ]:
def chunksFromText(text: str, chunk_size=500, overlap=50):
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks

test_text = "A" * 100 
test_chunks = chunksFromText(test_text, chunk_size=30, overlap=5)
print(f"Text length: {len(test_text)}")
print(f"Number of chunks: {len(test_chunks)}")
print(f"First chunk length: {len(test_chunks[0]) if test_chunks else 0}")

Text length: 100
Number of chunks: 4
First chunk length: 30


In [ ]:
def chunksFromPDF(vDB, path: str, startPage=0, endPage=None):
    reader = PdfReader(path)
    pages = reader.pages[startPage:endPage]

    print(f"📄 Processing {len(pages)} pages from PDF...")

    all_chunks = []

    for page_num, page in enumerate(tqdm(pages, desc="Extracting text"), startPage):
        text = page.extract_text()
        text = clean_text(text)

        if len(text) < 100:
            continue

        page_chunks = chunksFromText(text)
        page_chunks = [f"[Page {page_num+1}] {chunk}" for chunk in page_chunks]
        all_chunks.extend(page_chunks)

    vDB.addToDatabase(all_chunks)

    print(f"✅ Successfully processed PDF: {len(all_chunks)} chunks added to database")

Implement the RAG pipeline

In [ ]:
def generateAnswer(question: str, vDB, llm):
    relevant_chunks, scores = vDB.search(question, n_return=3)
    context = "\n\n".join(relevant_chunks)
    prompt = f"""
Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:
"""  
    result = llm(prompt, max_length=200, num_return_sequences=1)
    generated_text = result[0]['generated_text']
    answer = generated_text.split("Answer:")[-1].strip()

    return answer

In [ ]:
print("Logging in to Hugging Face Hub...")
try:
    login(token=HF_API_KEY)
    print("✅ Login successful!")
except Exception as e:
    print(f"⚠️ Skipping login (API key optional): {e}")

Logging in to Hugging Face Hub...
⚠️ Skipping login (API key optional): Invalid user token.


In [ ]:
print("Loading embedding model...")
vDB = VectorDB(EMBEDDING_MODEL)
print("✅ Vector Database ready!")

Loading embedding model...
✅ VectorDB initialized with embedding dimension: 384
✅ Vector Database ready!


In [ ]:
pdf_path = "TechNova_IT_Handbook.pdf"

if not os.path.exists(pdf_path):
    print(f"❌ PDF not found at {pdf_path}")
    print("Please make sure 'TechNova_IT_Handbook.pdf' is in the same folder as this notebook.")
else:
    print(f"📄 Found PDF: {pdf_path}")

    chunksFromPDF(vDB, pdf_path)

📄 Found PDF: TechNova_IT_Handbook.pdf
📄 Processing 21 pages from PDF...


Extracting text: 100%|██████████| 21/21 [00:00<00:00, 218.10it/s]


✅ Added 40 chunks. Total chunks: 40
✅ Successfully processed PDF: 40 chunks added to database


In [ ]:
print("Loading LLM model...")
print("⏳ This may take a few minutes...")

llm = pipeline("text2text-generation", model=LLM_MODEL)
print("✅ LLM loaded successfully!")

Loading LLM model...
⏳ This may take a few minutes...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ LLM loaded successfully!


In [ ]:
print("\n" + "="*50)
print("🤖 RAGBot is ready!")
print("Ask questions about the TechNova IT Handbook.")
print("Type 'exit' or 'quit' to stop.")
print("="*50 + "\n")

while True:
    # Get user input
    question = input("\n❓ Your question: ")

    # Check if user wants to exit
    if question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye! Thanks for using RAGBot!")
        break

    # Skip empty questions
    if not question.strip():
        continue

    # TODO: Generate and print the answer
    # Hint: answer = generateAnswer(question, vDB, llm)
    print("\n🤔 Thinking...\n")
    answer = generateAnswer(question, vDB, llm)

    print(f"💡 Answer: {answer}")
    print("\n" + "-"*50)


🤖 RAGBot is ready!
Ask questions about the TechNova IT Handbook.
Type 'exit' or 'quit' to stop.


❓ Your question: what's this book about?

🤔 Thinking...



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: [iii]

--------------------------------------------------

❓ Your question: How many pages does this book have

🤔 Thinking...



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: [Page 18]

--------------------------------------------------

❓ Your question: What do email and teams require?

🤔 Thinking...



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: (4).

--------------------------------------------------

❓ Your question: what is granted at onboarding?

🤔 Thinking...



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


💡 Answer: unanswerable

--------------------------------------------------

❓ Your question: exit

👋 Goodbye! Thanks for using RAGBot!


In [ ]:
test_question = "What is the password policy?"

print(f"Question: {test_question}\n")
answer = generateAnswer(test_question, vDB, llm)
print(f"Answer: {answer}")

Question: What is the password policy?



Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: No reuse of last 10 passwords; rotate every 365 days.
